# MovieLens 100K — EDA Day 1
Descriptive EDA only; model fitting and user histories must use train. Run all cells to export four figures. Outputs are intentionally cleared.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Run from repository root or notebooks/')
DATA = ROOT / 'data/processed'
FIGURES = ROOT / 'reports/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
ratings = pd.read_parquet(DATA / 'ratings.parquet')
movies = pd.read_parquet(DATA / 'movies.parquet')
print(f'{len(ratings):,} ratings / {ratings.user_id.nunique()} users / {ratings.movie_id.nunique()} movies')
print(f'Sparsity: {1 - len(ratings)/(ratings.user_id.nunique()*ratings.movie_id.nunique()):.2%}')

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / name, dpi=160)
    plt.close(fig)

## 1. Rating distribution
Counts for each explicit rating value, including all 100K observations.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ratings.rating.value_counts().reindex(range(1, 6), fill_value=0).plot.bar(ax=ax, rot=0)
ax.set(title='MovieLens 100K: rating distribution', xlabel='Rating', ylabel='Number of ratings')
save(fig, 'rating_distribution.png')

## 2. Ratings per user
Histogram of activity counts; this describes the selected dataset, not population activity.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ratings.groupby('user_id').size(), bins=35, edgecolor='white')
ax.set(title='MovieLens 100K: ratings per user', xlabel='Ratings per user', ylabel='Number of users')
save(fig, 'ratings_per_user.png')

## 3. Movie popularity long tail
Movies ranked by rating count. The y axis uses a logarithmic scale.

In [ ]:
counts = ratings.groupby('movie_id').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(counts) + 1), counts.to_numpy())
ax.set_yscale('log')
ax.set(title='MovieLens 100K: movie popularity long tail', xlabel='Movie rank by rating count', ylabel='Number of ratings (log scale)')
save(fig, 'ratings_per_movie_longtail.png')

## 4. Top genres by rating count
Each rating contributes to every genre assigned to its movie; counts can sum to more than 100K.

In [ ]:
genres = ['unknown', 'action', 'adventure', 'animation', 'children', 'comedy', 'crime', 'documentary', 'drama', 'fantasy', 'film_noir', 'horror', 'musical', 'mystery', 'romance', 'sci_fi', 'thriller', 'war', 'western']
joined = ratings[['movie_id']].merge(movies[['movie_id', *genres]], on='movie_id', validate='many_to_one')
top = joined[genres].sum().sort_values(ascending=False).head(10).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
top.plot.barh(ax=ax)
ax.set(title='MovieLens 100K: top 10 genres by ratings', xlabel='Number of ratings', ylabel='Genre')
save(fig, 'top_genres.png')
print('Saved four figures to', FIGURES)